In [1]:
import pandas as pd
import numpy as np
import requests
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from lime.lime_text import LimeTextExplainer
import warnings

warnings.filterwarnings('ignore')

TMDB_API_KEY = "8fbff5545ff86558e8b6fd669a4c8076" 

df = pd.read_csv('../data_clean/clean_masterdata.csv')
df = df.drop_duplicates(subset=['userId', 'movieId']).reset_index(drop=True)

df['user_cat'] = df['userId'].astype('category')
df['movie_cat'] = df['title'].astype('category')

unique_movies = df[['title', 'movie_cat', 'overview', 'genres']].drop_duplicates(subset=['movie_cat'])
unique_movies = unique_movies.sort_values(by='movie_cat').reset_index(drop=True)

unique_movies['mix'] = unique_movies['overview'].fillna('') + " " + unique_movies['genres'].fillna('')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(unique_movies['mix']) 

sparse_matrix = csr_matrix((df['rating'], (df['movie_cat'].cat.codes, df['user_cat'].cat.codes)))
svd = TruncatedSVD(n_components=50, random_state=42)
latent_matrix = svd.fit_transform(sparse_matrix) 

unique_movies['row_num'] = unique_movies['movie_cat'].cat.codes
indices = pd.Series(unique_movies['row_num'].values, index=unique_movies['title'].str.lower().str.strip())
indices = indices[~indices.index.duplicated(keep='first')]

sentiment_analyzer = pipeline("sentiment-analysis", device=0) 

explainer = LimeTextExplainer(class_names=['Negative', 'Positive'])


C:\Users\athul\Downloads\CINEIQ_PRJCT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 5429.99it/s]


In [13]:

def fetch_5_random_reviews(movie_title):
    try:
        search_url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={movie_title}"
        search_response = requests.get(search_url).json()
        if not search_response['results']: return []
            
        movie_id = search_response['results'][0]['id']
        reviews_url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?api_key={TMDB_API_KEY}"
        reviews_response = requests.get(reviews_url).json()
        
        all_reviews = reviews_response['results']
        if not all_reviews: return []
            
        num_to_grab = min(5, len(all_reviews))
        return [r['content'] for r in random.sample(all_reviews, num_to_grab)]
    except:
        return []

def get_sentiment_score(review_text):
    result = sentiment_analyzer(review_text[:1500])[0]
    return result['score'] if result['label'] == 'POSITIVE' else -result['score']


def lime_predictor(texts):
    results = sentiment_analyzer(texts, truncation=True, max_length=512)
    probs = []
    for res in results:
        if res['label'] == 'POSITIVE':
            probs.append([1 - res['score'], res['score']])
        else:
            probs.append([res['score'], 1 - res['score']])
    return np.array(probs)

def generate_lime_explanation(review_text):
    if len(review_text) < 10:
        return "Recommended based on strong  matching."
    
    exp = explainer.explain_instance(
        review_text[:500], 
        lime_predictor, 
        num_features=8, 
        num_samples=100
    )
    word_weights = exp.as_list()
    
    positive_words = []
    
    for raw_word, weight in word_weights:
        clean_word = str(raw_word).lower() 
        
        if weight > 0 and clean_word.isalpha() and len(clean_word) >= 5:
            positive_words.append(clean_word)
            
    if positive_words:
        formatted_words = ", ".join(positive_words)
        return f"Recommended based on keywords: {formatted_words}."
    else:
        return "Recommended based on strong matching."

In [14]:
def master_cineiq_search_optimized(title, content_weight=0.5, collab_weight=0.5):
    clean_title = title.lower().strip()
    if clean_title not in indices:
        return f"'{title}' is not in the database."
    
    idx = indices[clean_title]
    
    content_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    collab_scores = cosine_similarity(latent_matrix[idx].reshape(1, -1), latent_matrix).flatten()
    hybrid_scores = (content_scores * content_weight) + (collab_scores * collab_weight)
    
    sim_scores = sorted(list(enumerate(hybrid_scores)), key=lambda x: x[1], reverse=True)[1:16]
    
    final_output_data = []
    
    for i, (movie_idx, combined_score) in enumerate(sim_scores):
        candidate_title = unique_movies.loc[movie_idx, 'title']
        base_score = 15 - i 
        sentiment_modifier = 0.0
        
        real_reviews = fetch_5_random_reviews(candidate_title)
        
        if len(real_reviews) > 0:
            total_sentiment = sum([get_sentiment_score(r) for r in real_reviews])
            sentiment_modifier = total_sentiment / len(real_reviews)
            
        adjusted_score = base_score + (sentiment_modifier * 4.0)
        
        final_output_data.append({
            'score': adjusted_score,
            'title': candidate_title,
            'reviews_cache': real_reviews 
        })
        
    top_3_winners = sorted(final_output_data, key=lambda x: x['score'], reverse=True)[:3]
    
    
    for movie in top_3_winners:
        lime_explanation = "Recommended based on strong algorithmic matching."
        reviews = movie['reviews_cache']
        
        if len(reviews) > 0:
            review_scores = [(r, get_sentiment_score(r)) for r in reviews]
            review_scores.sort(key=lambda x: abs(x[1]), reverse=True)
            most_impactful_review = review_scores[0][0]
            
            lime_explanation = generate_lime_explanation(most_impactful_review[:500])
            
        movie['lime_report'] = lime_explanation
    
    print(f"TOP 3 RECOMMENDATIONS FOR '{title.upper()}'")

    
    for rank, movie in enumerate(top_3_winners, 1):
        print(f"{rank}. {movie['title']}")
        print(f"   {movie['lime_report']}\n")

master_cineiq_search_optimized("Inception")


🎬 CINEIQ: TOP 3 RECOMMENDATIONS FOR 'INCEPTION'

1. Interstellar
   Recommended based on keywords: christopher, great, science, produced, showcased.

2. The Dark Knight
   Recommended based on keywords: superb, again, thrilling, magnificent, unforgettable.

3. Shutter Island
   Recommended based on keywords: performances, reason, procrastinated.

